# Density Matrix Method

This notebook integrates the QuTiP master-equation approach into the project format.

What it does:
- supports `fock` and `coherent` initial states
- computes the mean particle number from `\u27e8n\u27e9`
- computes the variance from `\u27e8n^2\u27e9 - \u27e8n\u27e9^2`
- saves one CSV in the same format as the other methods

Example filename:
- `densityMatrix_fock_numOfParticles5_gamma1_time5_dt0p001_numOfSamples1_hilbertSize7.csv`


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve().parent
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)


In [ ]:
import matplotlib.pyplot as plt

try:
    import scienceplots  # noqa: F401
    plt.style.use(["science"])
    USING_SCIENCEPLOTS = True
except ImportError:
    USING_SCIENCEPLOTS = False
    print("scienceplots is not installed; using default matplotlib style.")

from bosonic_dissipation.config import DEFAULT_HILBERT_SIZE, resolve_hilbert_size
from bosonic_dissipation.density_matrix_method import run_density_matrix_and_save

global_defaults = {
    "initial_state_type": "fock",
    "num_of_particles": 1,
    "gamma": 1.0,
    "time": 5.0,
    "dt": 1e-3,
    "num_of_samples": 1,
    "hilbert_size": DEFAULT_HILBERT_SIZE,
}

density_matrix_config = {
    "initial_state_type": global_defaults["initial_state_type"],
    "num_of_particles": global_defaults["num_of_particles"],
    "gamma": global_defaults["gamma"],
    "time": global_defaults["time"],
    "dt": global_defaults["dt"],
    "num_of_samples": 1,
    "hilbert_size": global_defaults["hilbert_size"],
}

if density_matrix_config["hilbert_size"] is None:
    density_matrix_config["hilbert_size"] = resolve_hilbert_size(
        initial_state_type=density_matrix_config["initial_state_type"],
        num_of_particles=density_matrix_config["num_of_particles"],
        hilbert_size=density_matrix_config["hilbert_size"],
    )

print(f"Using scienceplots: {USING_SCIENCEPLOTS}")
density_matrix_config

## Runtime Block: Density Matrix

This mirrors your original QuTiP script, but now it follows the same save-and-plot pattern as the rest of the project.


In [ ]:
density_matrix_result, density_matrix_csv_path = run_density_matrix_and_save(
    OUTPUT_DIR,
    initial_state_type=density_matrix_config["initial_state_type"],
    num_of_particles=density_matrix_config["num_of_particles"],
    gamma=density_matrix_config["gamma"],
    time=density_matrix_config["time"],
    dt=density_matrix_config["dt"],
    num_of_samples=density_matrix_config["num_of_samples"],
    hilbert_size=density_matrix_config["hilbert_size"],
)

print(f"Backend used: {density_matrix_result.backend}")
print(f"Runtime (s): {density_matrix_result.runtime_seconds:.6f}")
print(f"Hilbert size: {density_matrix_result.hilbert_size}")
if density_matrix_result.coherent_alpha is not None:
    print(f"Coherent alpha: {density_matrix_result.coherent_alpha}")
print(f"Saved CSV: {density_matrix_csv_path}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

label_suffix = density_matrix_result.initial_state_type.capitalize()

axes[0].plot(
    density_matrix_result.time_values,
    density_matrix_result.mean_particle_number,
    label=f"Density Matrix ({label_suffix})",
)
axes[0].set_title("Mean Particle Number")
axes[0].set_xlabel("Time")
axes[0].set_ylabel("Mean Particle Number")
axes[0].legend()
axes[0].grid(True)

axes[1].plot(
    density_matrix_result.time_values,
    density_matrix_result.variance,
    label=f"Density Matrix ({label_suffix})",
    color="tab:orange",
)
axes[1].set_title("Variance")
axes[1].set_xlabel("Time")
axes[1].set_ylabel("Variance")
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

In [ ]:
density_matrix_result.time_values[:5], density_matrix_result.mean_particle_number[:5], density_matrix_result.variance[:5]